# llm_fake

> Offline stand-ins for Phase 10: `FakeClient` (the `messages.create` surface with scripted replies, used by every offline test) and `fakeReply` (a deterministic policy computed from the request text, used by the quick demo). Neither touches the network.

In [ ]:
#| default_exp llm_fake

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json
import re
from types import SimpleNamespace


class FakeClient:
    """Offline stand-in with the `messages.create` surface: `script(request) -> dict(text, stop, refusal, usage)` or a
    list of such dicts (or plain texts) used in turn. Usage defaults to a deterministic size estimate."""
    def __init__(self, script):
        self.script, self.requests = script, []
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **req):
        self.requests.append(req)
        r = self.script(req) if callable(self.script) else self.script[(len(self.requests) - 1) % len(self.script)]
        r = dict(text=r) if isinstance(r, str) else dict(r)
        text = r.get('text', '')
        usage = {**dict(input_tokens=len(json.dumps(req)) // 4, cache_creation_input_tokens=0, cache_read_input_tokens=0,
                        output_tokens=max(1, len(text) // 4)), **r.get('usage', {})}
        details = SimpleNamespace(category=r.get('refusal')) if r.get('stop') == 'refusal' else None
        return SimpleNamespace(content=[SimpleNamespace(type='text', text=text)], stop_reason=r.get('stop', 'end_turn'),
                               stop_details=details, usage=SimpleNamespace(**usage))


def _h(s):
    return int(hashlib.sha256(s.encode()).hexdigest()[:8], 16)


def fakeReply(req):
    "Deterministic offline policy for quick runs: a function of the request text only."
    blocks = req['messages'][0]['content']
    o, h = blocks[-1]['text'], _h(blocks[-1]['text'])
    if 'estimates' in req['output_config']['format']['schema']['properties']:
        seen = re.findall(r'stayed at (\w+), earned (-?\d+\.\d+)', ' '.join(b['text'] for b in blocks))
        return json.dumps(dict(estimates=[dict(location=n, value=float(v)) for n, v in seen]))
    props = req['output_config']['format']['schema']['properties']
    if 'vote' in props:                                     # the hidden-profile vote (Phase 13): one of the two names
        return json.dumps(dict(vote=props['vote']['enum'][h % 2], notes=f'h{h % 7}'))
    if 'kind' in props:                                     # a hidden-profile post: mostly one of the own facts
        ids = re.findall(r'^(F\d+): ', o, re.M)
        if ids and h % 4:
            return json.dumps(dict(kind='fact', fact=ids[h % len(ids)], text='', notes=f'h{h % 7}'))
        return json.dumps(dict(kind='opinion', fact='', text=f'I lean towards one option (h{h % 5}).', notes=f'h{h % 7}'))
    acts = props.get('action', {}).get('enum', ())
    op = re.search(r'Your open subtask: T(\d+), open', o)
    seen = re.search(r'(?:the board|the team): T(\d+): (x mod \d+ = -?\d+)', o)
    if 'ask' in acts and op and h % 5 == 1:                 # Phase 13 options (only when in the schema): ask, check
        return json.dumps(dict(action='ask', target=f'T{op.group(1)}', answer=0, clue='', notes=f'h{h % 7}'))
    if 'check' in acts and seen and h % 5 == 2:
        return json.dumps(dict(action='check', target=f'T{seen.group(1)}', answer=0, clue=seen.group(2), notes=f'h{h % 7}'))
    own = re.search(r'Your own clues: T(\d+): (x mod \d+ = \d+)', o)
    if own and h % 3 == 0:                                  # the clue board: sometimes share an own clue
        return json.dumps(dict(action='share', target=f'T{own.group(1)}', answer=0, clue=own.group(2), notes=f'h{h % 7}'))
    if 'You are at' in o:
        here, near = re.search(r'You are at (\w+)', o).group(1), re.search(r': (\w+) [\d.]+ s', o).group(1)
        go = h % 3 == 0
        return json.dumps(dict(action='go' if go else 'stay', target=near if go else '', notes=f'h{h % 7}',
                               reportLocation=here if h % 2 else '', reportValue=float(h % 5) / 4))
    m, first = re.search(r'Your open subtask: T(\d+), open', o), re.search(r'\): T(\d+)', o).group(1)
    return json.dumps(dict(action='submit' if m else 'open', target='' if m else f'T{first}', answer=h % 50,
                           claimTask=f'T{m.group(1) if m else first}', claimKind='working', notes=f'h{h % 7}'))

## Tests

In [ ]:
import json
from explRL.llm_fake import *
from explRL.llm_client import LLMClient
from explRL.llm_env import FORAGE_SCHEMA, TextForagingEnv, locationNames
from explRL.llm_board import TextBoardEnv
from explRL.llm_agent import BELIEF_SCHEMA
from explRL.harness_runs import benchWorld
from explRL.taskboard import TaskBoardEnv

# scripted replies: a list is used in turn (cycling), a function sees each request; stops, refusals and usage overrides
fc = FakeClient(['a', dict(text='b', stop='max_tokens'), dict(text='', stop='refusal', refusal='cyber', usage=dict(output_tokens=7))])
r = [fc.messages.create(model='m', messages=[]) for _ in range(4)]
assert [x.content[0].text for x in r] == ['a', 'b', '', 'a'] and [x.stop_reason for x in r] == ['end_turn', 'max_tokens', 'refusal', 'end_turn']
assert r[2].stop_details.category == 'cyber' and r[2].usage.output_tokens == 7 and r[0].stop_details is None
assert len(fc.requests) == 4 and FakeClient(lambda req: req['model']).messages.create(model='x').content[0].text == 'x'

# fakeReply gives schema-shaped, parseable actions for both tasks and for the belief question, deterministically
w = benchWorld(); w.reset(0, 1); env = TextForagingEnv(w, locationNames(64, 0))
c = LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
req = c.request('rules', [{'type': 'text', 'text': 'Now:\n' + env.observe(0, 0.0)}], FORAGE_SCHEMA)
a = json.loads(fakeReply(req))
assert set(a) == set(FORAGE_SCHEMA['required']) and env.parse(a, 0)[2] and fakeReply(req) == fakeReply(req)
b = TaskBoardEnv(0, 1); benv = TextBoardEnv(b)
a = json.loads(fakeReply(c.request('rules', [{'type': 'text', 'text': benv.observe(0, 0.0)}], benv.SCHEMA)))
assert a['action'] == 'open' and benv.parse(a, 0)[2]
hist = [{'type': 'text', 'text': f'Your past turns, oldest first:\nt=0.0 s: stayed at {env.names[3]}, earned 0.40.'}]
e = json.loads(fakeReply(c.request('rules', hist + [{'type': 'text', 'text': 'The run is over.'}], BELIEF_SCHEMA)))
assert e == {'estimates': [{'location': env.names[3], 'value': 0.4}]}
# the hidden-profile schemas (Phase 13): a post names one of the prompt's fact ids or is an opinion; a vote is one of the names
P = {'type': 'object', 'properties': {k: {'type': 'string'} for k in ('kind', 'fact', 'text', 'notes')}}
V = {'type': 'object', 'properties': {'vote': {'type': 'string', 'enum': ['Avery', 'Blake']}, 'notes': {'type': 'string'}}}
posts = [json.loads(fakeReply(c.request('rules', [{'type': 'text', 'text': f'Round 1, case {s}.\nYour facts:\nF1: a\nF2: b'}], P)))
         for s in range(40)]
assert {p['kind'] for p in posts} == {'fact', 'opinion'} and {p['fact'] for p in posts if p['kind'] == 'fact'} == {'F1', 'F2'}
votes = [json.loads(fakeReply(c.request('rules', [{'type': 'text', 'text': f'Vote {s}'}], V)))['vote'] for s in range(40)]
assert set(votes) == {'Avery', 'Blake'} and votes == [json.loads(fakeReply(c.request('rules', [{'type': 'text', 'text': f'Vote {s}'}], V)))['vote'] for s in range(40)]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()